# Create Propositions with FLAN-T5 (Batch)

Este notebook transforma os `cleaned_text` de um arquivo JSON em proposições jurídicas usando `transformers` com `google/flan-t5-base` e processamento em lote para reduzir tempo de execução.

In [ ]:
%pip install -q transformers accelerate sentencepiece torch tqdm

In [ ]:
import json
import math
import os
import time
import torch

from google.colab import drive
from pathlib import Path
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


## Configuração

In [ ]:
BASE_PATH = '/content/drive/MyDrive/tcc/data/'

CACHE_DIR = os.path.join(BASE_PATH, 'bm25/bm25-marked-paragraphs-to-llm/cache')

# Ajuste estes caminhos conforme seu ambiente (local ou Colab)
PARAGRAPHS_FILE = os.path.join(CACHE_DIR, 'query_marked_paragraphs.json')
PROPOSITIONS_FILE = os.path.join(CACHE_DIR, 'query_propositions_with_llm.json')

# Opcional: filtrar por doc_id especifico (None processa todos)
TARGET_DOC_ID = None

# Opcional: limitar quantidade total de paragrafos para teste rapido
MAX_PARAGRAPHS = None

# Modelo e parametros de inferencia
MODEL_NAME = 'google/flan-t5-base'
BATCH_SIZE = 16
MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 128
NUM_BEAMS = 2

PROMPT_TEMPLATE = """Convert the following fragment into a clear, concise legal
proposition. Keep it factual and neutral. Ensure the
proposition is understandable in isolation.
Example Fragment: On this point, the Respondent argues that the
Officer's conclusion that there was insufficient evidence
cannot be read in isolation and must be considered in the
context of the findings and summary of evidence prior to such
conclusion...
Example Proposition: The Court should not interfere with the Officer
's decision unless it is outside the range of acceptable
outcomes.
Now generate a proposition for this fragment: {fragment}
"""

print(f'INPUT_JSON_PATH: {PARAGRAPHS_FILE}')
print(f'OUTPUT_JSON_PATH: {PROPOSITIONS_FILE}')
print(f'MODEL_NAME: {MODEL_NAME}')
print(f'BATCH_SIZE: {BATCH_SIZE}')

INPUT_JSON_PATH: /content/drive/MyDrive/tcc/data/bm25/bm25-marked-paragraphs-to-llm/cache/query_marked_paragraphs.json
OUTPUT_JSON_PATH: /content/drive/MyDrive/tcc/data/bm25/bm25-marked-paragraphs-to-llm/cache/query_propositions_with_llm.json
MODEL_NAME: google/flan-t5-base
BATCH_SIZE: 16


## Funções de Preparação e Batch Inference

In [ ]:
def normalize_doc_id(value):
    return str(value).strip()


def build_prompt(fragment: str) -> str:
    return PROMPT_TEMPLATE.format(fragment=fragment)


def load_and_filter_rows(input_json_path: str, target_doc_id=None, max_paragraphs=None):
    input_path = Path(input_json_path)
    if not input_path.exists():
        raise FileNotFoundError(f'Arquivo nao encontrado: {input_path}')

    with input_path.open('r', encoding='utf-8') as f:
        rows = json.load(f)

    if not isinstance(rows, list):
        raise ValueError('Esperado JSON de entrada como lista de registros.')

    selected = []
    for row in rows:
        cleaned_text = str(row.get('cleaned_text', '')).strip()
        if not cleaned_text:
            continue

        if target_doc_id is not None and normalize_doc_id(row.get('doc_id', '')) != normalize_doc_id(target_doc_id):
            continue

        selected.append({
            'doc_id': row.get('doc_id'),
            'paragraph_id': row.get('paragraph_id'),
            'cleaned_text': cleaned_text,
        })

    if max_paragraphs is not None:
        selected = selected[:max_paragraphs]

    if not selected:
        raise ValueError('Nenhum paragrafo encontrado apos filtros aplicados.')

    print(f'Total de paragrafos selecionados: {len(selected)}')
    if target_doc_id is not None:
        print(f'Filtro de doc_id aplicado: {target_doc_id}')

    return selected


def generate_propositions_in_batches(records, model_name, batch_size=16, max_source_length=512, max_target_length=128, num_beams=2):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Device em uso: {device}')

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    model.to(device)
    model.eval()

    prompts = [build_prompt(item['cleaned_text']) for item in records]
    all_outputs = []

    start_time = time.perf_counter()

    total_batches = math.ceil(len(prompts) / batch_size)
    for batch_idx in tqdm(range(total_batches), desc='Gerando proposicoes', unit='batch'):
        start = batch_idx * batch_size
        end = min((batch_idx + 1) * batch_size, len(prompts))
        batch_prompts = prompts[start:end]

        inputs = tokenizer(
            batch_prompts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=max_source_length,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_target_length,
                num_beams=num_beams,
                do_sample=False,
                early_stopping=True,
            )

        batch_outputs = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        all_outputs.extend([text.strip() for text in batch_outputs])

    elapsed = time.perf_counter() - start_time

    results = []
    for rec, proposition in zip(records, all_outputs):
        results.append({
            'doc_id': rec['doc_id'],
            'paragraph_id': rec['paragraph_id'],
            'cleaned_text': rec['cleaned_text'],
            'proposition': proposition,
        })

    return results, elapsed

## Executar Geração e Salvar JSON

In [ ]:
records = load_and_filter_rows(
    input_json_path=PARAGRAPHS_FILE,
    target_doc_id=TARGET_DOC_ID,
    max_paragraphs=MAX_PARAGRAPHS,
)

results, elapsed = generate_propositions_in_batches(
    records=records,
    model_name=MODEL_NAME,
    batch_size=BATCH_SIZE,
    max_source_length=MAX_SOURCE_LENGTH,
    max_target_length=MAX_TARGET_LENGTH,
    num_beams=NUM_BEAMS,
)

payload = {
    'source_file': PARAGRAPHS_FILE,
    'target_doc_id': TARGET_DOC_ID,
    'model': MODEL_NAME,
    'batch_size': BATCH_SIZE,
    'total_fragments_found': len(records),
    'total_propositions_generated': len(results),
    'elapsed_seconds': elapsed,
    'elapsed_hms': time.strftime('%H:%M:%S', time.gmtime(elapsed)),
    'items': results,
}

output_path = Path(PROPOSITIONS_FILE)
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open('w', encoding='utf-8') as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print(f'Output salvo em: {output_path}')
print(f'Proposicoes geradas: {len(results)}')
print(f'Tempo total: {payload["elapsed_hms"]} ({elapsed:.2f}s)')

Total de paragrafos selecionados: 19442
Device em uso: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Gerando proposicoes:   0%|          | 0/1216 [00:00<?, ?batch/s]

Output salvo em: /content/drive/MyDrive/tcc/data/bm25/bm25-marked-paragraphs-to-llm/cache/query_propositions_with_llm.json
Proposicoes geradas: 19442
Tempo total: 01:10:22 (4222.75s)


In [ ]:
# Visualizacao rapida de exemplos
for i, item in enumerate(results[:3], start=1):
    print('\n' + '=' * 80)
    print(f'Exemplo {i}')
    print(f"doc_id: {item['doc_id']} | paragraph_id: {item['paragraph_id']}")
    print('Fragmento:')
    print(item['cleaned_text'][:350] + ('...' if len(item['cleaned_text']) > 350 else ''))
    print('Proposicao:')
    print(item['proposition'])


Exemplo 1
doc_id: 000002 | paragraph_id: 10
Fragmento:
The first issue is one of procedural fairness and must be reviewed according to the standard of correctness ( ; 344 N.R. 257; 2005 FCA 404).
Proposicao:
The first issue is procedural fairness and must be reviewed according to the standard of correctness.

Exemplo 2
doc_id: 000002 | paragraph_id: 11
Fragmento:
The second issue involves establishing the content and interpretation of Cameroonian law, and the case law recognizes that such issues constitute questions of fact that must be reviewed according to the standard of reasonableness ( ; 2008 FC 252; ; 2007 FC 364; ; 278 N.R. 127; 2001 FCA 311; and ).
Proposicao:
The second issue involves establishing the content and interpretation of Cameroonian law, and the case law recognizes that such issues constitute questions of fact that must be reviewed according to the standard of reasonableness.

Exemplo 3
doc_id: 000002 | paragraph_id: 12
Fragmento:
In applying the standard of reasona